# Etapa 8 · Evaluación única en prueba reservada

La puntuación se ejecutó una sola vez con modelos y umbrales congelados. Este cuaderno **no abre la prueba ni vuelve a ejecutar los modelos**; únicamente lee los resultados cerrados para hacerlos auditables.

## 1. Evidencia de cierre

Comprueba fecha, huellas, modelos y umbrales aplicados.

In [1]:
# ARCHIVO: cuaderno de lectura de la evaluación cerrada.
# BLOQUE 1: dependencias y rutas.
from pathlib import Path  # Maneja rutas portables.
import json  # Lee el estado cerrado.
import pandas as pd  # Lee tablas finales.
from IPython.display import display, Markdown  # Presenta resultados.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()  # Localiza la raíz al ejecutar desde proyecto o notebooks.
SALIDAS = RAIZ / 'reports/eda08_evaluacion'  # Localiza resultados sin acceder a la prueba.
estado = json.loads((SALIDAS / '00_estado_evaluacion.json').read_text(encoding='utf-8'))  # Lee el cierre persistente.
assert estado['estado'] == 'completada_y_cerrada'  # Exige que la evaluación esté cerrada.
display(estado)  # Muestra la evidencia de cierre.


{'estado': 'completada_y_cerrada',
 'fecha': '2026-09-30',
 'sha256_prueba': '1763f3a95dcade1340e4fb7abea2b72451892048d9787016726d58ef0d38473b',
 'sha256_predicciones': 'e20aa2fd19c8975954e25a199e094de1d47eaebe49deffa7d17a758a3ebc2c6c',
 'modelos_aplicados': ['logistica_predictiva_congelada.joblib',
  'boosting_congelado.joblib'],
 'umbrales_aplicados': {'base': 0.4999999999999998,
  'logistica': 0.6949999999999997,
  'boosting': 0.7049999999999997},
 'replicas_bootstrap_validas': 1000,
 'se_permite_repetir_puntuacion': False}

## 2. Desempeño fuera de muestra

Compara prevalencia, logística y boosting con ponderación anual.

In [2]:
# BLOQUE 2: desempeño global e intervalos.
globales = pd.read_csv(SALIDAS / '01_metricas_prueba_globales.csv')  # Lee estimaciones puntuales.
intervalos = pd.read_csv(SALIDAS / '02_metricas_prueba_ic95.csv')  # Lee intervalos por UPM.
seleccion = intervalos.loc[intervalos['metrica'].isin(['roc_auc', 'pr_auc', 'brier', 'exactitud_balanceada'])]  # Selecciona resultados principales.
display(globales.round(4))  # Presenta todas las métricas.
display(seleccion.round(4))  # Presenta estimaciones e IC.


,modelo,n,upm,umbral_congelado,roc_auc,pr_auc,brier,log_loss,sensibilidad,especificidad,precision,exactitud,exactitud_balanceada,tasa_predicha_positiva,intercepto_calibracion,pendiente_calibracion,convergencia_calibracion
0,base,7791,1269,0.500,0.5000,0.6873,0.2150,0.6214,1.0000,0.0000,0.6873,0.6873,0.5000,1.0000,-0.0226,0.9813,True
1,logistica,7791,1269,0.695,0.6736,0.8125,0.1982,0.5808,0.5909,0.6740,0.7994,0.6169,0.6324,0.5081,0.0261,0.9298,True
2,boosting,7791,1269,0.705,0.6842,0.8244,0.1953,0.5740,0.6148,0.6268,0.7836,0.6185,0.6208,0.5392,0.1341,0.7609,True


,modelo,metrica,estimacion,ic95_inferior,ic95_superior,replicas_validas,metodo_ic
0,base,roc_auc,0.5000,0.5000,0.5000,1000,bootstrap UPM dentro de estrato
1,base,pr_auc,0.6873,0.6586,0.7170,1000,bootstrap UPM dentro de estrato
2,base,brier,0.2150,0.2034,0.2262,1000,bootstrap UPM dentro de estrato
8,base,exactitud_balanceada,0.5000,0.5000,0.5000,1000,bootstrap UPM dentro de estrato
10,logistica,roc_auc,0.6736,0.6445,0.7024,1000,bootstrap UPM dentro de estrato
11,logistica,pr_auc,0.8125,0.7833,0.8420,1000,bootstrap UPM dentro de estrato
12,logistica,brier,0.1982,0.1854,0.2100,1000,bootstrap UPM dentro de estrato
18,logistica,exactitud_balanceada,0.6324,0.6038,0.6595,1000,bootstrap UPM dentro de estrato
20,boosting,roc_auc,0.6842,0.6544,0.7137,1000,bootstrap UPM dentro de estrato
21,boosting,pr_auc,0.8244,0.7941,0.8522,1000,bootstrap UPM dentro de estrato


## 3. ¿Mejora el modelo de árboles?

Los intervalos pareados permiten distinguir una ventaja puntual de una mejora respaldada por la prueba.

In [3]:
# BLOQUE 3: comparación pareada.
comparacion = pd.read_csv(SALIDAS / '03_comparacion_pareada_modelos.csv')  # Lee boosting menos logística.
display(comparacion.round(4))  # Presenta diferencias y sus IC.


,comparacion,metrica,diferencia,ic95_inferior,ic95_superior,replicas_validas,direccion_favorable
0,boosting menos logistica,roc_auc,0.0106,-0.0097,0.0303,1000,mayor
1,boosting menos logistica,pr_auc,0.0119,-0.0012,0.0237,1000,mayor
2,boosting menos logistica,brier,-0.0028,-0.0083,0.0032,1000,menor
3,boosting menos logistica,log_loss,-0.0068,-0.0195,0.0065,1000,menor
4,boosting menos logistica,exactitud_balanceada,-0.0117,-0.0388,0.0145,1000,mayor


## 4. Diferencias entre grupos

Se revisan discriminación, error probabilístico, sensibilidad y especificidad con el mismo umbral global congelado.

In [4]:
# BLOQUE 4: auditoría por sexo y área.
grupos = pd.read_csv(SALIDAS / '05_metricas_prueba_por_grupo.csv')  # Lee desempeño por grupo.
brechas = pd.read_csv(SALIDAS / '06_brechas_desempeno_ic95.csv')  # Lee diferencias pareadas.
display(grupos.round(4))  # Presenta métricas de cada grupo.
display(brechas.round(4))  # Presenta brechas e IC.


,modelo,variable_grupo,codigo_grupo,n,upm,roc_auc,pr_auc,brier,log_loss,sensibilidad,especificidad,precision,exactitud,exactitud_balanceada,tasa_predicha_positiva
0,logistica,p02,1,3619,1023,0.6638,0.8406,0.1816,0.5433,0.6860,0.5743,0.8178,0.6565,0.6302,0.6173
1,logistica,p02,2,4172,1106,0.6654,0.7766,0.2125,0.6131,0.4974,0.7380,0.7756,0.5827,0.6177,0.4139
2,logistica,area,1,6916,1032,0.6656,0.8168,0.1962,0.5764,0.6163,0.6383,0.7981,0.6229,0.6273,0.5397
3,logistica,area,2,875,237,0.7009,0.7722,0.2118,0.6118,0.3858,0.8655,0.8155,0.5746,0.6256,0.2869
4,boosting,p02,1,3619,1023,0.6762,0.8532,0.1816,0.5432,0.6919,0.5293,0.8037,0.6489,0.6106,0.6335
5,boosting,p02,2,4172,1106,0.6771,0.7891,0.2072,0.6006,0.5390,0.6894,0.7596,0.5924,0.6142,0.4580
6,boosting,area,1,6916,1032,0.6710,0.8272,0.1951,0.5733,0.6247,0.5886,0.7789,0.6138,0.6066,0.5605
7,boosting,area,2,875,237,0.7461,0.8028,0.1966,0.5787,0.5347,0.8316,0.8303,0.6516,0.6832,0.3905


,modelo,variable,brecha,metrica,diferencia,ic95_inferior,ic95_superior,replicas_validas
0,logistica,p02,Mujer menos hombre,roc_auc,0.0016,-0.0445,0.0414,1000
1,logistica,p02,Mujer menos hombre,brier,0.0309,0.0172,0.0452,1000
2,logistica,p02,Mujer menos hombre,sensibilidad,-0.1887,-0.2421,-0.1334,1000
3,logistica,p02,Mujer menos hombre,especificidad,0.1637,0.0954,0.2268,1000
4,logistica,area,Rural menos urbana,roc_auc,0.0353,-0.0378,0.1023,1000
5,logistica,area,Rural menos urbana,brier,0.0155,-0.0053,0.0398,1000
6,logistica,area,Rural menos urbana,sensibilidad,-0.2306,-0.3111,-0.1524,1000
7,logistica,area,Rural menos urbana,especificidad,0.2272,0.1287,0.3219,1000
8,boosting,p02,Mujer menos hombre,roc_auc,0.0010,-0.0516,0.0567,1000
9,boosting,p02,Mujer menos hombre,brier,0.0256,0.0064,0.0446,1000


## 5. Conclusión

El boosting muestra una ventaja puntual pequeña en discriminación, pero su intervalo pareado incluye cero. La logística conserva mejor calibración y mayor exactitud balanceada. Ambos modelos presentan menor sensibilidad para mujeres y personas del área rural con los umbrales globales congelados.